# 04 -- Multi-macro chipathon flow: counter + 4-bit ALU

End-to-end walkthrough of hardening two RTL modules (`counter` and
`alu_macro`) as **independent LibreLane macros**, then stitching both
into the chipathon-2026 workshop padring. The example illustrates:

1. **RTL verification** with cocotb (counter freeze/wrap, ALU truth
   table).
2. **Standalone macro hardening** (LibreLane Classic flow, no padring).
3. **Post-synthesis gate-level simulation** of the hardened netlists
   against the gf180 stdcell behavioural models.
4. **Integration**: dropping the two hardened GDS/LEF/lib views into
   `chip_top` and invoking the full Chip flow on the workshop slot
   with multi-corner STA, Magic DRC, Netgen LVS, KLayout DRC, antenna.

Everything runs inside the `hpretl/iic-osic-tools:next` Docker
container (`gf180`, bind-mount `~/eda/designs <-> /foss/designs`).
The repo-level `scripts/bootstrap_container.sh` starts it if not up.

**Workspace isolation.** This notebook writes two dedicated dirs:

| Host path                                       | Purpose                        |
|-------------------------------------------------|--------------------------------|
| `~/eda/designs/multimacro_demo/`                | RTL + TB + macro LibreLane runs|
| `~/eda/designs/multimacro_chipathon/template/`  | Dedicated padring fork copy    |

It NEVER touches `~/eda/designs/chipathon_padring/template/` (that is
the baseline used by notebook 03 and must stay clean).

**Credits.** This notebook consumes
<https://github.com/Mauricio-xx/chipathon-2026-gf180mcu-padring>, itself
a derivation of <https://github.com/wafer-space/gf180mcu-project-template>
plus the workshop pad layout from
<https://github.com/JuanMoya/padring_gf180>. All Apache-2.0.

**Runtime budget** (modern 32-core workstation):

| Step                     | Runtime    |
|--------------------------|------------|
| cocotb RTL sim           | ~15 s      |
| cocotb GL sim (optional) | ~15 s      |
| harden counter macro     | ~1.5-3 min |
| harden alu_macro         | ~1.5-3 min |
| chip-top flow (workshop) | ~60-90 min |

All `RUN_*` flags default to `False`; flip them one by one when ready.


## Step 0 -- configuration


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import textwrap

# ---- flags -------------------------------------------------------------
RUN_CLONE_FORK    = False   # clone chipathon-2026 padring fork (~200 KB)
RUN_CLONE_PDK     = False   # clone wafer-space GF180 PDK @ 1.8.0 (~500 MB)
RUN_STAGE_FILES   = False   # copy rtl/ + tb/ + librelane/ into bind-mount
RUN_COCOTB        = False   # ~15 s, counter + alu tests (RTL)
RUN_HARDEN_COUNTER = False  # ~1.5-3 min
RUN_HARDEN_ALU    = False   # ~1.5-3 min
RUN_GLSIM         = False   # ~15 s, post-synth GL sim (optional)
RUN_PATCH_TOP     = False   # write chip_core_multi.sv + patched config.yaml into fork copy
RUN_CHIP_TOP      = False   # ~60-90 min, SLOT=workshop full chip flow

# ---- container ---------------------------------------------------------
CONTAINER_NAME = 'gf180'

# ---- upstream references (Apache-2.0) ---------------------------------
FORK_URL     = 'https://github.com/Mauricio-xx/chipathon-2026-gf180mcu-padring.git'
PDK_FORK_URL = 'https://github.com/wafer-space/gf180mcu.git'
PDK_FORK_TAG = '1.8.0'

# ---- host paths --------------------------------------------------------
# The fork gets its own dedicated path for this example so we never
# corrupt another workspace (e.g. the `chipathon_padring/template/`
# baseline used by notebooks 03 or earlier runs).
HOST_WORKSPACE  = Path.home() / 'eda' / 'designs'
HOST_FORK       = HOST_WORKSPACE / 'multimacro_chipathon' / 'template'
HOST_PDK        = HOST_FORK / 'gf180mcu'
HOST_MULTIMACRO = HOST_WORKSPACE / 'multimacro_demo'

# Repo-relative source dirs. The notebook lives at
# examples/04_counter_alu_multimacro/<nb>.ipynb. Modern Jupyter sets
# __file__ when the kernel runs the cell; older kernels do not, so we
# fall back to cwd. Either way we verify rtl/ tb/ librelane/ exist
# next to NB_DIR; otherwise raise here (config time) instead of
# leaving a silent failure for RUN_STAGE_FILES later.
try:
    NB_DIR = Path(__file__).resolve().parent
except NameError:
    NB_DIR = Path.cwd().resolve()

for _sub in ('rtl', 'tb', 'librelane'):
    if not (NB_DIR / _sub).is_dir():
        raise RuntimeError(
            f"NB_DIR={NB_DIR} does not contain {_sub}/. "
            "Launch this notebook from "
            "examples/04_counter_alu_multimacro/ so rtl/ tb/ librelane/ "
            "resolve correctly."
        )

SRC_RTL     = NB_DIR / 'rtl'
SRC_TB      = NB_DIR / 'tb'
SRC_LRL     = NB_DIR / 'librelane'

# ---- container paths ---------------------------------------------------
CONTAINER_FORK       = '/foss/designs/multimacro_chipathon/template'
CONTAINER_MULTIMACRO = '/foss/designs/multimacro_demo'

# ---- PDK identifiers ---------------------------------------------------
PDK_NAME     = 'gf180mcuD'
STD_CELL_LIB = 'gf180mcu_fd_sc_mcu7t5v0'


def run_or_print(cmd, do_it, *, shell_on_container=False, timeout=None):
    if shell_on_container:
        print(f"$ docker exec {CONTAINER_NAME} bash -lc '<script>'")
        print(textwrap.indent(cmd, '  | '))
    else:
        print('$ ' + ' '.join(str(x) for x in cmd))
    if not do_it:
        print('  (skipped -- flip the RUN_* flag to execute)\n')
        return None
    args = (
        ['docker', 'exec', CONTAINER_NAME, 'bash', '-lc', cmd]
        if shell_on_container else list(cmd)
    )
    proc = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    if proc.stdout.strip():
        print(proc.stdout[-4000:])
    if proc.returncode != 0 and proc.stderr.strip():
        print('STDERR (tail):')
        print(proc.stderr[-2000:])
    print(f'  returncode={proc.returncode}\n')
    return proc


print(f'Repo-relative sources:')
print(f'  rtl:       {SRC_RTL}')
print(f'  tb:        {SRC_TB}')
print(f'  librelane: {SRC_LRL}')
print()
print(f'Bind-mount destination: {HOST_MULTIMACRO}  (container: {CONTAINER_MULTIMACRO})')
print(f'Dedicated fork copy:    {HOST_FORK}  (container: {CONTAINER_FORK})')


## Step 1 -- prerequisites (container, fork, PDK)

The `gf180` container must be running. If `docker ps` shows no such
container, run `scripts/bootstrap_container.sh` at the repo root
(one-time `docker pull` + `docker run -d --name gf180 ...`).

This cell also clones the padring fork + the wafer-space PDK fork on
demand. Both steps are idempotent (no-op if already cloned).


In [ ]:
def ok(label, cond, detail=''):
    tag = 'OK ' if cond else '!! '
    print(f'{tag}{label}' + (f'  -- {detail}' if detail else ''))
    return cond

# Container alive?
proc = subprocess.run(
    ['docker', 'ps', '--filter', f'name={CONTAINER_NAME}', '--format', '{{.Names}}'],
    capture_output=True, text=True,
)
container_up = CONTAINER_NAME in proc.stdout
ok(f"Container '{CONTAINER_NAME}' running", container_up)

# Fork clone (optional auto-clone).
if not HOST_FORK.exists() or not (HOST_FORK / '.git').exists():
    HOST_WORKSPACE.mkdir(parents=True, exist_ok=True)
    clone_cmd = ['git', 'clone', '--depth', '1', FORK_URL, str(HOST_FORK)]
    run_or_print(clone_cmd, RUN_CLONE_FORK, timeout=180)
ok('Padring fork present',
   HOST_FORK.exists() and (HOST_FORK / 'librelane' / 'slots' / 'slot_workshop.yaml').exists(),
   str(HOST_FORK))

# PDK fork clone.
if not HOST_PDK.exists():
    clone_pdk = ['git', 'clone', '--depth', '1', '--branch', PDK_FORK_TAG,
                 PDK_FORK_URL, str(HOST_PDK)]
    run_or_print(clone_pdk, RUN_CLONE_PDK, timeout=600)
ok('wafer-space PDK fork present', HOST_PDK.exists(), str(HOST_PDK))


## Step 2 -- stage example files into the bind-mount

The example lives in the git-cloned repo, which may be anywhere on
your host filesystem. LibreLane (running inside the container) needs
to see the files under `/foss/designs/...`, so we copy them into
`~/eda/designs/multimacro_demo/`.


In [ ]:
if RUN_STAGE_FILES:
    if HOST_MULTIMACRO.exists():
        shutil.rmtree(HOST_MULTIMACRO)
    HOST_MULTIMACRO.mkdir(parents=True)
    for sub in ('rtl', 'tb', 'librelane'):
        src = NB_DIR / sub
        dst = HOST_MULTIMACRO / sub
        shutil.copytree(src, dst)
        n = sum(1 for _ in dst.rglob('*') if _.is_file())
        print(f'  {sub}/  -- {n} file(s)')
    (HOST_MULTIMACRO / 'build').mkdir(exist_ok=True)
    print(f'\nStaged at {HOST_MULTIMACRO}')
else:
    print(f'(dry-run) would stage:')
    for sub in ('rtl', 'tb', 'librelane'):
        print(f'  {NB_DIR / sub}  ->  {HOST_MULTIMACRO / sub}')


## Step 3 -- RTL verification (cocotb)

Runs two cocotb testbenches inside the container:
- `test-counter` -- reset, increment, enable freeze, 256-cycle wrap.
- `test-alu` -- all 8 ops across all 16 x 16 (a, b) pairs, 2048 checks.

Both use Icarus Verilog; `cocotb` + `iverilog` ship pre-installed in
`hpretl/iic-osic-tools:next`.


In [ ]:
cocotb_script = textwrap.dedent(f'''
    set -e
    cd {CONTAINER_MULTIMACRO}/tb
    make clean
    make test-counter
    make test-alu
''').strip()

run_or_print(cocotb_script, RUN_COCOTB, shell_on_container=True, timeout=300)


## Step 4 -- harden the counter macro (standalone Classic flow)

Runs `librelane librelane/counter_macro.yaml` to synth + PnR + signoff
a standalone 8-bit counter. The outputs land in
`build/counter/final/` and are the views `chip_top` will reference
under `MACROS:` later.

Runtime: ~3-5 min.


In [ ]:
harden_counter = textwrap.dedent(f'''
    set -e
    cd {CONTAINER_MULTIMACRO}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    librelane librelane/counter_macro.yaml \\
        --pdk {PDK_NAME} \\
        --pdk-root {CONTAINER_FORK}/gf180mcu \\
        --manual-pdk \\
        --save-views-to {CONTAINER_MULTIMACRO}/build/counter
''').strip()

run_or_print(harden_counter, RUN_HARDEN_COUNTER,
             shell_on_container=True, timeout=900)


## Step 5 -- harden alu_macro

Same pattern. `alu_macro.sv` is the registered wrapper (1 FF stage on
inputs + 1 on outputs) around the pure-combinational `alu.sv`; the
Classic flow wants a clock, hence the wrapper.

Runtime: ~3-5 min.


In [ ]:
harden_alu = textwrap.dedent(f'''
    set -e
    cd {CONTAINER_MULTIMACRO}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    librelane librelane/alu_macro.yaml \\
        --pdk {PDK_NAME} \\
        --pdk-root {CONTAINER_FORK}/gf180mcu \\
        --manual-pdk \\
        --save-views-to {CONTAINER_MULTIMACRO}/build/alu_macro
''').strip()

run_or_print(harden_alu, RUN_HARDEN_ALU,
             shell_on_container=True, timeout=900)


## Step 5b -- post-synthesis gate-level simulation (optional but recommended)

After Steps 4-5 produce `build/counter/nl/counter.nl.v` and
`build/alu_macro/nl/alu_macro.nl.v`, we re-run cocotb against those
synthesised netlists paired with the GF180 stdcell behavioural models
(`gf180mcu_fd_sc_mcu7t5v0.v` + `primitives.v`, shipped inside the
wafer-space PDK fork). This confirms the synthesis tool preserved
functional equivalence, independent of the yosys-internal equivalence
checks that LibreLane already runs as part of the Classic flow.

The TBs are the same ones that exercised the RTL in Step 3:

- `test_counter.py` drives `counter.nl.v` (DFF instances from gf180 +
  synthesised combinational logic).
- `test_alu_macro.py` drives `alu_macro.nl.v` (both input + output
  registers become concrete gf180 DFF primitives; the combinational
  ALU becomes stdcell gates).

The GL builds use separate `sim_build_*_gl/` directories so RTL and
GL caches do not collide, and compile with `-DFUNCTIONAL
-DUNIT_DELAY=#1` to activate the functional behaviour + unit-delay
timing of the gf180 cells. Post-layout SDF timing is out of scope
for this example (requires exporting an SDF from OpenROAD and
annotating via `$sdf_annotate`).

This step is optional: a failure here indicates synthesis is
non-equivalent to RTL, which is rare for designs this small but
always worth catching before integration.


In [ ]:
# Optional GL sim. Set to True after Steps 4-5 have produced the
# synth netlists under build/{counter,alu_macro}/nl/.
RUN_GLSIM = False

glsim_script = textwrap.dedent(f"""
    set -e
    cd {CONTAINER_MULTIMACRO}/tb
    make test-counter-gl
    make test-alu-gl
""").strip()

run_or_print(glsim_script, RUN_GLSIM, shell_on_container=True, timeout=600)


## Step 6 -- patch chip_core + chip-top config

Three things happen here:

1. `chip_core_multi.sv` is copied over the fork's `src/chip_core.sv`
   (same module name; this is the chip-top's new core).
2. `counter.sv`, `alu.sv`, `alu_macro.sv` are copied into the fork's
   `src/` so the Chip-flow synthesiser can find them.
3. The fork's `librelane/config.yaml` is augmented with `MACROS:` and
   `PDN_MACRO_CONNECTIONS:` entries pointing at the build outputs
   from Steps 4-5.

All operations are idempotent. Re-run to refresh after re-hardening.


In [ ]:
def patch_top():
    try:
        import yaml
    except ImportError as e:
        raise RuntimeError(
            "PyYAML is not available in the host kernel. "
            "Install it (e.g. `pip install pyyaml`) before flipping "
            "RUN_PATCH_TOP=True. The container side does not need it."
        ) from e

    # 1. Copy chip_core_multi over the fork's chip_core.sv. Only this
    #    one file lands in src/ -- counter and alu_macro are hard
    #    macros placed via MACROS below, which blackboxes them for
    #    synthesis via their vh: entries. Parameters are NOT overridden
    #    on the instantiations (the hardened .nl.v carries no params).
    rtl_dst = HOST_FORK / 'src'
    shutil.copy2(HOST_MULTIMACRO / 'rtl' / 'chip_core_multi.sv', rtl_dst / 'chip_core.sv')
    print(f'Copied chip_core_multi.sv  ->  {rtl_dst}/chip_core.sv')

    # 2. Load fork's librelane/config.yaml and augment.
    cfg_path = HOST_FORK / 'librelane' / 'config.yaml'
    cfg = yaml.safe_load(cfg_path.read_text())

    # VERILOG_FILES stays minimal: only chip_top + (patched) chip_core.
    cfg['VERILOG_FILES'] = [
        'dir::../src/chip_top.sv',
        'dir::../src/chip_core.sv',
    ]

    # GF180 multi-corner STA key names. Provide one .lib per corner.
    # A single `*` mapping would silently collapse multi-corner STA to
    # a single corner.
    CORNERS = [
        'nom_tt_025C_5v00', 'nom_ss_125C_4v50', 'nom_ff_n40C_5v50',
        'min_tt_025C_5v00', 'min_ss_125C_4v50', 'min_ff_n40C_5v50',
        'max_tt_025C_5v00', 'max_ss_125C_4v50', 'max_ff_n40C_5v50',
    ]

    build = HOST_MULTIMACRO / 'build'

    # --save-views-to <dir> writes <dir>/{gds,lef,nl,lib/<corner>}/<design>.*
    # Paths below mirror that layout (no extra 'final/' subdir).
    def macro_entry(name, loc):
        base = build / name
        lib_map = {
            corner: [str(base / 'lib' / corner / f'{name}__{corner}.lib')]
            for corner in CORNERS
        }
        return {
            'gds': [str(base / 'gds' / f'{name}.gds')],
            'lef': [str(base / 'lef' / f'{name}.lef')],
            'vh':  [str(base / 'nl'  / f'{name}.nl.v')],
            'lib': lib_map,
            'instances': {f'i_chip_core.{loc["inst"]}': {
                'location': loc['xy'], 'orientation': 'N',
            }},
        }

    cfg.setdefault('MACROS', {})
    cfg['MACROS']['counter']   = macro_entry('counter',   {'inst': 'u_counter', 'xy': [1000, 1000]})
    cfg['MACROS']['alu_macro'] = macro_entry('alu_macro', {'inst': 'u_alu',     'xy': [1800, 1000]})

    # PDN_MACRO_CONNECTIONS = List[str] of "<regex> <vdd_net> <vss_net>
    # <macro_vdd_pin> <macro_vss_pin>".  LibreLane v3.0.2 rejects any
    # dict-shaped entry with "Refusing to automatically convert value
    # to string".
    cfg['PDN_MACRO_CONNECTIONS'] = [
        '.*u_counter.* VDD VSS VDD VSS',
        '.*u_alu.*     VDD VSS VDD VSS',
    ]

    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, default_flow_style=False))
    print(f'Patched {cfg_path}')

if RUN_PATCH_TOP:
    patch_top()
else:
    print('(dry-run) would copy chip_core_multi.sv over src/chip_core.sv')
    print('(dry-run) would add counter + alu_macro to MACROS (9 corners each) in librelane/config.yaml')
    print('(dry-run) would set PDN_MACRO_CONNECTIONS to string entries (LibreLane v3 schema)')


## Step 7 -- run the chip-top flow (SLOT=workshop)

Same `make librelane` invocation the fork already supports: floorplan
the padring, place the two user macros + chip_id + wafer.space logo,
PDN, global/detail routing, Magic + KLayout DRC, Netgen LVS, antenna,
STA.

Runtime: ~35-45 min on a modern laptop (Magic DRC dominates).


In [ ]:
chip_top_script = textwrap.dedent(f"""
    set -e
    cd {CONTAINER_FORK}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    make librelane SLOT=workshop PDK={PDK_NAME} PDK_ROOT={CONTAINER_FORK}/gf180mcu
""").strip()

run_or_print(chip_top_script, RUN_CHIP_TOP,
             shell_on_container=True, timeout=7200)


## Step 8 -- metrics and render

The chip-top build writes `final/metrics.csv` (one row per signoff
metric) and `final/render/chip_top.png`. A clean chip reports zero on
every `*_error__count`, `*_vio__count`, `*_violating__nets` row.


In [ ]:
import csv

metrics_path = HOST_FORK / 'final' / 'metrics.csv'
wanted = [
    ('design__die__area',                          'die area (um2)'),
    ('design__instance__count',                    'instance count'),
    ('design__instance__count__stdcell',           '  stdcells'),
    ('design__instance__count__class:macro',       '  macros (counter+alu+id+logo)'),
    ('design__instance__count__padcells',          '  pad cells'),
    ('magic__drc_error__count',                    'Magic DRC errors'),
    ('klayout__drc_error__count',                  'KLayout DRC errors'),
    ('design__lvs_error__count',                   'Netgen LVS errors'),
    ('antenna__violating__nets',                   'Antenna violations'),
    ('timing__setup_vio__count',                   'setup viols (all corners)'),
    ('timing__hold_vio__count',                    'hold viols  (all corners)'),
    ('power__total',                               'total power (W)'),
]

if not metrics_path.exists():
    print(f'metrics.csv not found: {metrics_path}')
    print('Did Step 7 complete?  Set RUN_CHIP_TOP = True and re-run.')
else:
    found = {}
    with metrics_path.open() as fh:
        for row in csv.reader(fh):
            if row and row[0] in dict(wanted):
                found[row[0]] = row[1] if len(row) > 1 else ''
    print(f'{"metric":45s} {"value":>15s}')
    print('-' * 63)
    for k, label in wanted:
        print(f'  {label:43s} {found.get(k, "(missing)"):>15s}')

    any_bad = False
    for k in ('magic__drc_error__count', 'klayout__drc_error__count',
              'design__lvs_error__count', 'antenna__violating__nets',
              'timing__setup_vio__count', 'timing__hold_vio__count'):
        v = found.get(k, '')
        try:
            if int(v) != 0:
                any_bad = True
        except ValueError:
            pass
    print()
    print('SIGNOFF:', 'CLEAN (all zero)' if not any_bad else 'VIOLATIONS PRESENT')


In [ ]:
from IPython.display import Image, display

png_path = HOST_FORK / 'final' / 'render' / 'chip_top.png'
gds_path = HOST_FORK / 'final' / 'gds'    / 'chip_top.gds'

if png_path.exists():
    print(f'Render: {png_path}')
    display(Image(str(png_path)))
else:
    print(f'No render PNG at {png_path}')
    if gds_path.exists():
        print(f'GDS: {gds_path}')
        print(f'Open on host: klayout {gds_path}')


## Where to go next

- **Tune placement.** `u_counter` sits at `[1000, 1000]`, `u_alu` at
  `[1800, 1000]`. Move them or change orientation (`N/S/E/W/FN/FE/...`)
  by editing the notebook's Step 6 merge logic -- no re-hardening
  needed.
- **Swap in a real macro** (SRAM, opamp, SAR ADC DAC). Harden it
  with the Classic flow the same way we did for `counter` and
  `alu_macro`, then append an entry under `MACROS:` with the same
  `gds/lef/vh/lib/instances` shape. PDN_MACRO_CONNECTIONS regex-matches
  against the hierarchical instance path.
- **Per-macro DRC/LVS signoff.** Steps 4-5 already run the full Classic
  signoff on each macro in isolation. `build/counter/final/metrics.csv`
  and `build/alu_macro/final/metrics.csv` carry the per-block numbers.
- **Compare to `03_rtl2gds_chipathon_use.ipynb`.** That notebook
  replaces only `chip_core.sv`; this one also touches `MACROS:` and
  `PDN_MACRO_CONNECTIONS:`.

Cleanup:

```bash
rm -rf ~/eda/designs/multimacro_demo
rm -rf ~/eda/designs/chipathon_padring/template/librelane/runs
docker stop gf180
```
